# TRUEWATCH — PP-OCRv5 Devanagari fine-tune on synthetic Nepali plates

Fine-tunes the pretrained `devanagari_PP-OCRv5_mobile_rec` recognition head on the
>= 20,000-sample synthetic Nepali plate corpus (`datasets/plates/gen_plates.py`,
`degrade.py`). MEASUREMENTS.md section 4 records that the pretrained model reads a
*synthetic* plate correctly (0.989/0.990) but only about half the characters on a
*real* plate photo, so fine-tuning on this corpus is required to approach the
slide 5 target of 85% end-to-end Nepali plate recognition.

**Run this on Kaggle, GPU enabled (Settings > Accelerator > GPU T4 x2 or P100), with
the packaged dataset attached as a private Kaggle Dataset.** Build that dataset
locally first:

```bash
python datasets/plates/gen_plates.py --count 20500 --out /path/to/scratch/plates
python edge/anpr/scripts/package_plates_for_kaggle.py \
    --source /path/to/scratch/plates --out edge/anpr/kaggle_upload --owner <kaggle-username>
kaggle datasets create -p edge/anpr/kaggle_upload --dir-mode zip
```

Then in a Kaggle Notebook: **Add Data > Your Datasets >
`<owner>/truewatch-nepali-plates-synthetic`**. This notebook was NOT run by the agent
that wrote it and nothing was uploaded on its behalf — running it, and uploading its
output checkpoint, is a manual step the orchestrator performs with Kaggle credentials.

Budget: this notebook targets Kaggle's free GPU quota (~4 h/week is generous for this
job; the fine-tune itself should finish in under an hour on a T4 for a 20k-sample,
single-line recognition head at 96x320 input).

## 0. Environment check

In [ ]:
import subprocess, sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("Python:", sys.version)

In [ ]:
# PaddlePaddle GPU build + PaddleOCR. Apache-2.0, matching edge/anpr's recognition
# engine so the exported checkpoint drops straight into edge/anpr/recognise.py.
%pip install -q -U paddlepaddle-gpu paddleocr pyyaml

## 1. Locate the attached dataset and the pretrained checkpoint

In [ ]:
from pathlib import Path

# Kaggle mounts an attached dataset under /kaggle/input/<dataset-slug>/.
# Adjust DATASET_SLUG if `package_plates_for_kaggle.py --title` was run with a
# different --title (the slug is the lowercased, space-to-hyphen title).
DATASET_SLUG = "truewatch-nepali-plates-synthetic"
DATA_ROOT = Path("/kaggle/input") / DATASET_SLUG
assert DATA_ROOT.is_dir(), f"expected the attached dataset at {DATA_ROOT}; check Add Data"

IMAGES_DIR = DATA_ROOT / "images"
TRAIN_LABELS = DATA_ROOT / "rec_gt_train.txt"
VAL_LABELS = DATA_ROOT / "rec_gt_val.txt"
CHARSET = DATA_ROOT / "charset.txt"
for p in (IMAGES_DIR, TRAIN_LABELS, VAL_LABELS, CHARSET):
    assert p.exists(), f"missing {p} — did package_plates_for_kaggle.py finish?"

print("train lines:", sum(1 for _ in TRAIN_LABELS.open(encoding="utf-8")))
print("val lines:", sum(1 for _ in VAL_LABELS.open(encoding="utf-8")))
print("charset size:", sum(1 for _ in CHARSET.open(encoding="utf-8")))

In [ ]:
# Fetch the pretrained checkpoint PaddleOCR ships for devanagari_PP-OCRv5_mobile_rec
# so this run fine-tunes FROM it rather than training from scratch. Using the
# high-level API once, off a 1-pixel dummy image, is the simplest way to trigger
# PaddleX's model download/cache and print where it landed.
from paddleocr import TextRecognition
import numpy as np

_bootstrap = TextRecognition(model_name="devanagari_PP-OCRv5_mobile_rec")
PRETRAINED_DIR = Path.home() / ".paddlex" / "official_models" / "devanagari_PP-OCRv5_mobile_rec"
assert PRETRAINED_DIR.is_dir(), PRETRAINED_DIR
print("pretrained checkpoint cached at", PRETRAINED_DIR)
del _bootstrap

## 2. PaddleOCR recognition training config

PaddleOCR's recognition trainer (`PaddleOCR/tools/train.py` from the
`PaddlePaddle/PaddleOCR` repository, Apache-2.0) takes a YAML config. This cell
clones the trainer and writes a config that fine-tunes from the pretrained
Devanagari checkpoint on the packaged label files above.

In [ ]:
import subprocess

REPO_DIR = Path("/kaggle/working/PaddleOCR")
if not REPO_DIR.is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/PaddlePaddle/PaddleOCR.git", str(REPO_DIR)],
        check=True,
    )
%pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml

OUTPUT_DIR = Path("/kaggle/working/output/devanagari_finetune")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "Global": {
        "model_name": "devanagari_PP-OCRv5_mobile_rec_finetune",
        "use_gpu": True,
        "epoch_num": 60,
        "log_smooth_window": 20,
        "print_batch_step": 50,
        "save_model_dir": str(OUTPUT_DIR),
        "save_epoch_step": 5,
        "eval_batch_step": [0, 500],
        "pretrained_model": str(PRETRAINED_DIR / "inference"),
        "checkpoints": None,
        "save_inference_dir": str(OUTPUT_DIR / "inference"),
        "use_visualdl": False,
        "character_dict_path": str(CHARSET),
        "max_text_length": 12,
        "infer_mode": False,
        "use_space_char": True,
    },
    "Optimizer": {
        "name": "Adam",
        "beta1": 0.9,
        "beta2": 0.999,
        "lr": {"name": "Cosine", "learning_rate": 0.0005, "warmup_epoch": 2},
        "regularizer": {"name": "L2", "factor": 3.0e-05},
    },
    "Architecture": {
        "model_type": "rec",
        "algorithm": "SVTR_LCNet",
    },
    "Loss": {"name": "MultiLoss", "loss_config_list": [{"CTCLoss": None}]},
    "PostProcess": {"name": "CTCLabelDecode"},
    "Metric": {"name": "RecMetric", "main_indicator": "acc"},
    "Train": {
        "dataset": {
            "name": "SimpleDataSet",
            "data_dir": str(IMAGES_DIR),
            "label_file_list": [str(TRAIN_LABELS)],
            "transforms": [
                {"DecodeImage": {"img_mode": "BGR", "channel_first": False}},
                {"RecAug": None},
                {"CTCLabelEncode": None},
                {"RecResizeImg": {"image_shape": [3, 48, 320]}},
                {"KeepKeys": {"keep_keys": ["image", "label", "length"]}},
            ],
        },
        "loader": {"shuffle": True, "batch_size_per_card": 128, "drop_last": True, "num_workers": 4},
    },
    "Eval": {
        "dataset": {
            "name": "SimpleDataSet",
            "data_dir": str(IMAGES_DIR),
            "label_file_list": [str(VAL_LABELS)],
            "transforms": [
                {"DecodeImage": {"img_mode": "BGR", "channel_first": False}},
                {"CTCLabelEncode": None},
                {"RecResizeImg": {"image_shape": [3, 48, 320]}},
                {"KeepKeys": {"keep_keys": ["image", "label", "length"]}},
            ],
        },
        "loader": {"shuffle": False, "batch_size_per_card": 128, "drop_last": False, "num_workers": 2},
    },
}

CONFIG_PATH = Path("/kaggle/working/devanagari_finetune.yml")
CONFIG_PATH.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(CONFIG_PATH.read_text(encoding="utf-8"))

## 3. Fine-tune

Note the real time budget: this cell can run for a while. Kaggle notebooks time out
at 12h/9h depending on accelerator; `epoch_num: 60` with `save_epoch_step: 5` leaves
resumable checkpoints in `OUTPUT_DIR` if it needs to be re-run from `checkpoints:`.

In [ ]:
import subprocess

result = subprocess.run(
    ["python", str(REPO_DIR / "tools" / "train.py"), "-c", str(CONFIG_PATH)],
    cwd=str(REPO_DIR),
)
print("train.py exit code:", result.returncode)
assert result.returncode == 0, "training failed — check the log above before continuing"

## 4. Export the inference model

`edge/anpr/recognise.py` loads a checkpoint directory via `ANPR_DEVANAGARI_MODEL_DIR`
(paddleocr's `TextRecognition(model_dir=...)`), which expects an exported inference
model directory (`inference.pdmodel`/`inference.pdiparams`-style layout), not a raw
training checkpoint.

In [ ]:
export_result = subprocess.run(
    [
        "python", str(REPO_DIR / "tools" / "export_model.py"),
        "-c", str(CONFIG_PATH),
        "-o", f"Global.pretrained_model={OUTPUT_DIR}/best_accuracy",
        f"Global.save_inference_dir={OUTPUT_DIR}/inference",
    ],
    cwd=str(REPO_DIR),
)
print("export exit code:", export_result.returncode)
assert export_result.returncode == 0

## 5. Evaluate the fine-tuned model with edge/anpr/evaluate.py's own metric

Re-uses the SAME exact-match/CER/distance-bucket code the pretrained baseline in
`edge/anpr/results/ANPR_METRICS.md` was measured with, so the two numbers are
directly comparable — not a different metric computed by the training framework.

In [ ]:
import os

os.environ["ANPR_DEVANAGARI_MODEL_DIR"] = str(OUTPUT_DIR / "inference")

# This repository's edge/anpr/ is not on the Kaggle image; clone the branch that
# has it (or upload edge/anpr/*.py as notebook "Add Data > Upload" files and adjust
# the path) before running this cell.
EDGE_ANPR_DIR = Path("/kaggle/working/truewatch/edge/anpr")
assert EDGE_ANPR_DIR.is_dir(), (
    f"expected the repository's edge/anpr/ checked out at {EDGE_ANPR_DIR}; "
    "clone the feat/phase5-anpr-face branch into /kaggle/working/truewatch first"
)

eval_result = subprocess.run(
    [
        "python", "evaluate.py",
        "--labels", str(VAL_LABELS),
        "--images-root", str(DATA_ROOT),
        "--out", "/kaggle/working/finetuned_eval.json",
    ],
    cwd=str(EDGE_ANPR_DIR),
)
print("evaluate.py exit code:", eval_result.returncode)

## 6. Download the checkpoint and the eval JSON

Kaggle Notebooks expose `/kaggle/working/` as downloadable output. Take:

- `output/devanagari_finetune/inference/` — the fine-tuned checkpoint. Install it
  wherever the edge service runs and point `ANPR_DEVANAGARI_MODEL_DIR` at it.
- `finetuned_eval.json` — the exact-match/CER/bucket numbers to paste into
  `edge/anpr/results/ANPR_METRICS.md`'s "after fine-tune" section, replacing the
  placeholder there. Do NOT overwrite the pretrained-baseline numbers already
  recorded — both must stay, clearly labelled, per the plan.